# 🔁 Retries, Error Handling & Snowflake Credentials with Prefect

---

## 🤔 Why Is This Extra Important for Snowflake?

Snowflake has network round-trips, warehouse cold-start time, and credit costs.  
Proper error handling means:
- No redundant Snowflake warehouse starts
- No silent failures with data quality
- Secure credential handling in all environments

---

## 💻 Example 1: Load Snowflake Credentials from Prefect Secrets

In [ ]:
import subprocess, os
from prefect import flow, task
from prefect.blocks.system import Secret

DBT_PROJECT_DIR = "/Users/aviraljain/Downloads/python advanced/my_etl_project"

def load_snowflake_env_from_secrets() -> dict:
    """
    Load Snowflake credentials from Prefect Secrets blocks.
    Falls back to environment variables if secrets not saved.
    """
    env = os.environ.copy()
    secret_map = {
        "SNOWFLAKE_ACCOUNT":   "snowflake-account",
        "SNOWFLAKE_USER":      "snowflake-user",
        "SNOWFLAKE_PASSWORD":  "snowflake-password",
        "SNOWFLAKE_WAREHOUSE": "snowflake-warehouse",
        "SNOWFLAKE_DATABASE":  "snowflake-database",
    }
    for env_key, block_name in secret_map.items():
        try:
            env[env_key] = Secret.load(block_name).get()
        except Exception:
            pass  # Fall back to whatever is in os.environ
    return env


def run_dbt(command: str, extra_flags: list = None) -> str:
    cmd = ["dbt"] + command.split()
    cmd += ["--project-dir", DBT_PROJECT_DIR, "--profiles-dir", DBT_PROJECT_DIR]
    if extra_flags: cmd += extra_flags
    result = subprocess.run(cmd, capture_output=True, text=True,
                            env=load_snowflake_env_from_secrets())
    print(result.stdout[-600:])
    if result.returncode != 0:
        raise RuntimeError(f"dbt failed: {result.stderr[-400:]}")
    return result.stdout

print("Secure credential loading defined ✅")

---

## 💻 Example 2: Retry Logic for Snowflake Timeouts

In [ ]:
# Snowflake warehouse cold-start can take ~10-30 seconds
# Set retry_delay_seconds >= 30 to give warehouse time to resume

@task(name="dbt seed → Snowflake", retries=1, retry_delay_seconds=30)
def dbt_seed():
    run_dbt("seed")
    print("✅ Seeds loaded into Snowflake")


@task(name="dbt run → Snowflake", retries=2, retry_delay_seconds=30)
def dbt_run(target: str = "dev", full_refresh: bool = False):
    # Adjust retry_delay_seconds=30 — Snowflake warehouse may need time to warm up
    extra = ["--full-refresh", "--target", target] if full_refresh else ["--target", target]
    run_dbt("run", extra_flags=extra)
    print(f"✅ dbt run complete on Snowflake (target={target})")


@task(name="dbt test → Snowflake", retries=0)  # No retry: failed test = real data problem
def dbt_test(target: str = "dev") -> dict:
    """Run dbt tests. Return dict instead of crashing — let flow decide."""
    cmd = ["dbt", "test", "--target", target,
           "--project-dir", DBT_PROJECT_DIR,
           "--profiles-dir", DBT_PROJECT_DIR]
    result = subprocess.run(cmd, capture_output=True, text=True,
                            env=load_snowflake_env_from_secrets())
    print(result.stdout[-600:])

    passed = result.returncode == 0
    if not passed:
        print("⚠️ Some dbt tests failed on Snowflake — check data quality!")
    return {"status": "passed" if passed else "failed", "returncode": result.returncode}


print("Retry-aware tasks defined ✅")

---

## 💻 Example 3: on_failure / on_completion Hooks for Alerting

In [ ]:
def alert_failure(flow, flow_run, state):
    """Triggered automatically if the flow fails."""
    print(f"🔴 SNOWFLAKE PIPELINE FAILED: {flow.name}")
    print(f"   State: {state.name} | Message: {state.message}")
    # In production, send Slack / PagerDuty / email alert here

def alert_success(flow, flow_run, state):
    """Triggered automatically if the flow succeeds."""
    print(f"🟢 SNOWFLAKE PIPELINE SUCCEEDED: {flow.name}")


@flow(
    name="dbt Snowflake (Error Handling)",
    log_prints=True,
    on_failure=[alert_failure],
    on_completion=[alert_success]
)
def dbt_snowflake_flow(target: str = "dev", full_refresh: bool = False):
    dbt_seed()
    dbt_run(target=target, full_refresh=full_refresh)
    test_result = dbt_test(target=target)

    if test_result["status"] == "failed":
        print("⚠️ Pipeline complete but Snowflake data quality issues exist!")
    else:
        print("✅ Snowflake pipeline complete — all tests passed!")

dbt_snowflake_flow(target="dev")

---

## 🏭 Summary

| Feature | Snowflake consideration | Prefect setting |
|---|---|---|
| Retry delay | Warehouse cold-start ~30s | `retry_delay_seconds=30` |
| Credentials | Never hardcode | Use `Prefect Secret` blocks |
| Test failures | Might be data issue, not infra | Return dict, don't raise |
| Failure hooks | Alert before Snowflake credits wasted | `on_failure=[...]` |

---

## ⚠️ Common Beginners' Mistakes

In [ ]:
mistakes = [
    ("retry_delay_seconds=5 for Snowflake",  "Too fast — warehouse needs 20-30s to resume from suspension"),
    ("retries=2 on dbt test",               "Data quality failures are real problems, not transient errors"),
    ("No Prefect Secrets for credentials",  "Hardcoded passwords in profiles.yml are a security risk"),
    ("No on_failure hook",                  "You won't know pipeline failed unless you check Prefect Cloud UI"),
]
for mistake, fix in mistakes:
    print(f"❌ {mistake}")
    print(f"✅ Fix: {fix}\n")